### Chapter 7 Exercises from Del Prado Textbook

In [1]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, make_scorer

In [9]:
# Made some fake synthetic data
np.random.seed(42)

n = 3000

# Simulate returns with weak autocorrelation/regime behavior
eps = np.random.normal(0, 0.01, n)
ret = np.zeros(n)

for t in range(1, n):
    ret[t] = 0.15 * ret[t-1] + eps[t]

close = pd.Series(100 * np.exp(np.cumsum(ret)))

X = pd.DataFrame(index=close.index)

returns = close.pct_change()

X["ret_1"] = returns.shift(1)
X["ret_2"] = returns.shift(2)
X["ret_3"] = returns.shift(3)

X["vol_10"] = returns.rolling(10).std()
X["vol_20"] = returns.rolling(20).std()

X["mom_5"] = close / close.shift(5) - 1
X["mom_10"] = close / close.shift(10) - 1

X["ma_5"] = close.rolling(5).mean()
X["ma_20"] = close.rolling(20).mean()
X["ma_ratio"] = X["ma_5"] / X["ma_20"] - 1

# Predict whether the 5-period future return is positive
future_ret = close.shift(-5) / close - 1
y = (future_ret > 0).astype(int)

data = X.join(y.rename("y")).dropna()

X = data.drop(columns=["y"])
y = data["y"]

print(X.shape)
print(y.value_counts())

t1 = pd.Series(X.index + 5, index=X.index)
valid = t1 < X.index.max()

X = X.loc[valid]
y = y.loc[valid]
t1 = t1.loc[valid]

(2980, 10)
y
1    1576
0    1404
Name: count, dtype: int64


### 7.1

Shuffling is intended to make each fold representative of the full dataset by randomly mixing observations before splitting. This is appropriate when observations are IID. In finance, however, observations are ordered in time and are often serially dependent. Randomly shuffling the data allows training folds to contain observations from the future relative to the test fold. It can also place overlapping labels from the same return path in both the training and testing sets. Therefore, shuffling creates information leakage and produces overly optimistic performance estimates.

### 7.2

In [7]:
# (a)
rf = RandomForestClassifier(n_estimators=100, max_depth=None, min_samples_leaf=5, random_state=42, n_jobs=-1)

cv_no_shuffle = KFold(n_splits=10, shuffle=False)

scores_no_shuffle = cross_val_score(rf, X, y, cv=cv_no_shuffle, scoring="accuracy", n_jobs=-1)

print("10-fold CV without shuffling")
print("Scores:", scores_no_shuffle)
print("Mean accuracy:", scores_no_shuffle.mean())
print("Std accuracy:", scores_no_shuffle.std())

for scoring in ["accuracy", "precision", "recall", "f1"]:
    scores = cross_val_score(rf, X, y, cv=cv_no_shuffle, scoring=scoring, n_jobs=-1)
    print(f"{scoring}: mean={scores.mean():.4f}, std={scores.std():.4f}")



10-fold CV without shuffling
Scores: [0.59060403 0.41610738 0.53355705 0.55704698 0.56711409 0.47651007
 0.51677852 0.48993289 0.45637584 0.52348993]
Mean accuracy: 0.512751677852349
Std accuracy: 0.050887355938972784
accuracy: mean=0.5128, std=0.0509
precision: mean=0.5302, std=0.0763
recall: mean=0.6181, std=0.2372
f1: mean=0.5377, std=0.1767


In [8]:
# (b)

cv_shuffle = KFold(n_splits=10, shuffle=True, random_state=42)

scores_shuffle = cross_val_score(rf, X, y, cv=cv_shuffle, scoring="accuracy", n_jobs=-1)

print("10-fold CV with shuffling")
print("Scores:", scores_shuffle)
print("Mean accuracy:", scores_shuffle.mean())
print("Std accuracy:", scores_shuffle.std())

10-fold CV with shuffling
Scores: [0.7147651  0.66778523 0.65100671 0.67114094 0.67114094 0.63422819
 0.67449664 0.67785235 0.63087248 0.68120805]
Mean accuracy: 0.6674496644295302
Std accuracy: 0.023076417781146576


(c) The shuffled and unshuffled CV results differ because financial data is not IID. In the shuffled case, observations from different time periods are randomly mixed, so the training set may contain observations that are temporally close to, or even later than, the test observations. This makes the prediction problem easier and usually inflates the estimated performance. Without shuffling, each test fold is a more coherent time block, so the model is tested on a less contaminated and more realistic sample.

(d) Shuffling leaks information because it breaks the temporal structure of the dataset. Training folds can contain observations from the future relative to the test fold. In addition, when labels overlap, a training observation and a test observation may be functions of the same future return path. Therefore, even if the exact test label is not included in training, closely related information used to determine that label may already appear in the training set.

### 7.3

In [10]:
class PurgedKFold:
    def __init__(self, n_splits=10, t1=None, pct_embargo=0.01):
        self.n_splits = n_splits
        self.t1 = t1
        self.pct_embargo = pct_embargo

    def split(self, X):
        indices = np.arange(X.shape[0])
        test_splits = np.array_split(indices, self.n_splits)

        embargo = int(X.shape[0] * self.pct_embargo)

        for test_indices in test_splits:
            test_start = test_indices[0]
            test_end = test_indices[-1]

            train_indices = indices.copy()
            test_start_time = X.index[test_start]
            test_end_time = self.t1.iloc[test_end]

            train_t0 = X.index
            train_t1 = self.t1

            overlap = (
                ((train_t0 >= test_start_time) & (train_t0 <= test_end_time)) |
                ((train_t1 >= test_start_time) & (train_t1 <= test_end_time)) |
                ((train_t0 <= test_start_time) & (train_t1 >= test_end_time))
            )

            train_indices = indices[~overlap]

            # Apply embargo after the test set
            embargo_start = test_end + 1
            embargo_end = min(test_end + 1 + embargo, X.shape[0])

            embargo_indices = indices[embargo_start:embargo_end]

            train_indices = np.setdiff1d(train_indices, embargo_indices)

            yield train_indices, test_indices

In [11]:
rf = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=42, n_jobs=-1)

pkf = PurgedKFold(n_splits=10, t1=t1,pct_embargo=0.01)

scores = []

for train_idx, test_idx in pkf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    scores.append({
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0)
    })

scores = pd.DataFrame(scores)

print(scores)
print("\nMean performance:")
print(scores.mean())
print("\nStd performance:")
print(scores.std())

   accuracy  precision    recall        f1
0  0.526846   0.554054  0.522293  0.537705
1  0.416107   0.406639  0.759690  0.529730
2  0.557047   0.674074  0.508380  0.579618
3  0.523490   0.554113  0.766467  0.643216
4  0.562290   0.630542  0.699454  0.663212
5  0.511785   0.529412  0.740506  0.617414
6  0.488215   0.507177  0.683871  0.582418
7  0.481481   0.457627  0.805970  0.583784
8  0.491582   0.566929  0.428571  0.488136
9  0.515152   0.666667  0.027397  0.052632

Mean performance:
accuracy     0.507399
precision    0.554723
recall       0.594260
f1           0.527786
dtype: float64

Std performance:
accuracy     0.041924
precision    0.086237
recall       0.236565
f1           0.175103
dtype: float64


### 7.4

Besides leakage, k-fold CV fails because financial data is non-stationary and not IID. The distribution changes over time, so folds are not exchangeable. A model that performs well on one historical regime may fail in another. Therefore, ordinary CV can give misleading estimates of future performance even if training and testing sets are carefully separated.